# Wine Production Analysis

Analyzing global wine production patterns by country — identifying major producers, production trends, and geographic distribution of the world's wine industry.

## Project Overview

Wine production is a significant global agricultural industry, with deep cultural and economic importance in many regions. This notebook analyzes wine production data by country to understand:

- Which countries dominate wine production
- How production volumes have changed over time
- Regional production patterns and market concentration
- Emerging and declining wine-producing nations

## Learning Objectives

1. Work with geographic and temporal production data
2. Calculate market share and concentration metrics
3. Create comparative visualizations across countries
4. Analyze production trends using time-series techniques
5. Handle Excel data formats in pandas
6. Build insights from agricultural/economic data

## Business / Research Problem

**Core question:** How is global wine production distributed geographically, and what trends are shaping the industry?

Understanding production patterns helps wine industry stakeholders, investors, agricultural policymakers, and trade analysts make informed decisions.

## Why This Analysis Matters

- **Economic significance:** Wine is a major export commodity for many countries
- **Climate impact:** Wine production is sensitive to climate change, making trends informative
- **Market understanding:** Production volumes influence global wine pricing and trade
- **Data skills:** Practice with real-world geographic and temporal data

## Dataset Overview

The dataset contains wine production statistics by country, with production volumes across multiple years/periods.

**Expected format:** Countries as rows, years/periods as columns, production volumes in relevant units (hectoliters or tonnes).

## Dataset Source and License Notes

- **Source:** [Kaggle — Wine Production by Country](https://www.kaggle.com/datasets/shitalgaikwad123/wine-production-by-country)
- **License:** Check Kaggle for current license terms
- **Original data:** Based on OIV (International Organisation of Vine and Wine) or similar sources

## Environment Setup

In [ ]:
!pip install -q pandas numpy matplotlib seaborn scipy openpyxl kaggle

## Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

print("All imports loaded.")

## Configuration / Constants

In [ ]:
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

DATA_DIR = 'data'

## Dataset Download

Download from Kaggle. Local file also available as fallback.

In [ ]:
import os
os.makedirs(DATA_DIR, exist_ok=True)

!kaggle datasets download -d shitalgaikwad123/wine-production-by-country -p {DATA_DIR} --unzip --force

# Find data files
data_file = None
for f in os.listdir(DATA_DIR):
    if f.endswith(('.csv', '.xlsx', '.xls')):
        data_file = os.path.join(DATA_DIR, f)
        break

# Fallback to local file
if data_file is None:
    for f in os.listdir('.'):
        if f.endswith(('.xlsx', '.xls', '.csv')) and 'wine' in f.lower():
            data_file = f
            break

print(f"Data file: {data_file}")

## Data Loading

In [ ]:
# Load based on file type
if data_file.endswith(('.xlsx', '.xls')):
    df = pd.read_excel(data_file)
else:
    df = pd.read_csv(data_file)

print(f"Dataset shape: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"\nColumns: {list(df.columns)}")
df.head(10)

## Data Validation Checks

In [ ]:
print("=== Data Types ===")
print(df.dtypes)

print("\n=== Missing Values ===")
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(1)
print(pd.DataFrame({'Count': missing, '%': missing_pct})[missing > 0])

print(f"\nDuplicates: {df.duplicated().sum()}")
print(f"\n=== Descriptive Stats ===")
df.describe()

## Data Cleaning

Reshape the data if needed and ensure numeric columns are properly typed.

In [ ]:
# Identify the country column
country_col = None
for col in df.columns:
    if df[col].dtype == 'object' and df[col].nunique() > 5:
        country_col = col
        break

if country_col is None:
    country_col = df.columns[0]

print(f"Country column: {country_col}")
print(f"Number of countries: {df[country_col].nunique()}")

# Identify year/numeric columns
year_cols = [c for c in df.columns if c != country_col]
print(f"Data columns: {year_cols[:10]}...")

# Convert to numeric
for col in year_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Fill NaN with 0 for countries that didn't produce in certain years
df[year_cols] = df[year_cols].fillna(0)

# Remove any 'World' or 'Total' rows
df = df[~df[country_col].str.lower().str.contains('total|world|other', na=False)]

print(f"\nCleaned shape: {df.shape}")

## Exploratory Data Analysis

### Overall Production Landscape

In [ ]:
# Calculate total and average production per country
df['Total'] = df[year_cols].sum(axis=1)
df['Average'] = df[year_cols].mean(axis=1)

# Sort by total production
df = df.sort_values('Total', ascending=False).reset_index(drop=True)

print("=== Top 10 Wine Producers (All Time) ===")
top10 = df.head(10)[[country_col, 'Total', 'Average']]
print(top10.to_string(index=False))

# Market share of top 10
total_global = df['Total'].sum()
top10_share = df.head(10)['Total'].sum() / total_global * 100
print(f"\nTop 10 countries produce {top10_share:.1f}% of total wine")

In [ ]:
# Top producers bar chart
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

top15 = df.head(15)
axes[0].barh(range(len(top15)), top15['Total'].values, color='#722f37')
axes[0].set_yticks(range(len(top15)))
axes[0].set_yticklabels(top15[country_col].values)
axes[0].invert_yaxis()
axes[0].set_title('Top 15 Wine Producers (Total Production)')
axes[0].set_xlabel('Total Production')

# Pie chart of top 5 + Others
top5 = df.head(5)
others = df.iloc[5:]['Total'].sum()
labels = list(top5[country_col].values) + ['Others']
sizes = list(top5['Total'].values) + [others]
colors = ['#722f37', '#c9b037', '#2ecc71', '#3498db', '#e74c3c', '#95a5a6']
axes[1].pie(sizes, labels=labels, autopct='%1.1f%%', colors=colors, startangle=90)
axes[1].set_title('Global Wine Production Share')

plt.tight_layout()
plt.show()

## Univariate Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution of total production
axes[0].hist(df['Total'], bins=30, edgecolor='black', alpha=0.7, color='#722f37')
axes[0].set_title('Distribution of Total Production by Country')
axes[0].set_xlabel('Total Production')

# Log distribution
log_total = np.log10(df['Total'].clip(lower=1))
axes[1].hist(log_total, bins=20, edgecolor='black', alpha=0.7, color='#c9b037')
axes[1].set_title('Log₁₀ Total Production Distribution')
axes[1].set_xlabel('log₁₀(Production)')

plt.tight_layout()
plt.show()

print(f"Production statistics:")
print(f"  Countries: {len(df)}")
print(f"  Mean: {df['Total'].mean():,.0f}")
print(f"  Median: {df['Total'].median():,.0f}")
print(f"  Std: {df['Total'].std():,.0f}")

## Bivariate / Multivariate Analysis

### Production trends over time

In [ ]:
# Time series for top producers
top5_countries = df.head(5)[country_col].values

fig, ax = plt.subplots(figsize=(14, 6))
for country in top5_countries:
    row = df[df[country_col] == country][year_cols].values.flatten()
    ax.plot(year_cols, row, marker='o', markersize=3, linewidth=2, label=country)

ax.set_title('Wine Production Trends — Top 5 Producers')
ax.set_xlabel('Year / Period')
ax.set_ylabel('Production')
ax.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Global total over time
global_total = df[year_cols].sum()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(global_total.index, global_total.values, marker='o', linewidth=2, color='#722f37')
ax.fill_between(global_total.index, global_total.values, alpha=0.2, color='#722f37')
ax.set_title('Global Wine Production Over Time')
ax.set_ylabel('Total Production')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Feature-Specific Insights

In [ ]:
# Production volatility (coefficient of variation)
df['CV'] = df[year_cols].std(axis=1) / df[year_cols].mean(axis=1).clip(lower=0.001)
df_filtered = df[df['Total'] > df['Total'].median()]

print("Most volatile producers (above-median total, by CV):")
volatile = df_filtered.nlargest(10, 'CV')[[country_col, 'Total', 'CV']]
print(volatile.round(3).to_string(index=False))

print("\nMost stable producers:")
stable = df_filtered.nsmallest(10, 'CV')[[country_col, 'Total', 'CV']]
print(stable.round(3).to_string(index=False))

In [ ]:
# Compare first and last periods to see growth/decline
if len(year_cols) >= 2:
    first_col = year_cols[0]
    last_col = year_cols[-1]
    
    df['Change'] = df[last_col] - df[first_col]
    df['Change_Pct'] = ((df[last_col] - df[first_col]) / df[first_col].clip(lower=0.001)) * 100
    
    # Top growers
    growers = df[df[first_col] > 0].nlargest(10, 'Change_Pct')
    print("Top 10 fastest growing producers:")
    print(growers[[country_col, first_col, last_col, 'Change_Pct']].round(1).to_string(index=False))
    
    # Top decliners
    print("\nTop 10 declining producers:")
    decliners = df[df[first_col] > 0].nsmallest(10, 'Change_Pct')
    print(decliners[[country_col, first_col, last_col, 'Change_Pct']].round(1).to_string(index=False))

## Statistical Checks / Hypothesis Testing

In [ ]:
alpha = 0.05

# 1. Is production log-normally distributed?
log_prod = np.log10(df['Total'].clip(lower=1))
stat, p = stats.shapiro(log_prod.sample(min(50, len(log_prod)), random_state=42))
print(f"1. Log-normality of production: W={stat:.4f}, p={p:.4f}")
print(f"   Log production is {'approximately normal' if p >= alpha else 'NOT normal'}")

# 2. Spearman correlation between first and last period rankings
if len(year_cols) >= 2:
    valid = df[(df[first_col] > 0) & (df[last_col] > 0)]
    r, p = stats.spearmanr(valid[first_col], valid[last_col])
    print(f"\n2. Rank correlation (first vs last period): ρ={r:.3f}, p={p:.2e}")
    print(f"   Rankings {'ARE' if p < alpha else 'are NOT'} significantly correlated")

# 3. Concentration ratio
top3_share = df.head(3)['Total'].sum() / total_global * 100
top5_share = df.head(5)['Total'].sum() / total_global * 100
top10_share2 = df.head(10)['Total'].sum() / total_global * 100
print(f"\n3. Market concentration:")
print(f"   CR3 (top 3): {top3_share:.1f}%")
print(f"   CR5 (top 5): {top5_share:.1f}%")
print(f"   CR10 (top 10): {top10_share2:.1f}%")

# 4. Herfindahl-Hirschman Index
market_shares = df['Total'] / total_global
hhi = (market_shares ** 2).sum() * 10000
print(f"   HHI: {hhi:.0f} ({'Concentrated' if hhi > 1500 else 'Competitive'})")

## Visual Analysis

In [ ]:
# Heatmap of production for top 10 countries over time
top10_data = df.head(10).set_index(country_col)[year_cols]

fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(top10_data, cmap='YlOrRd', ax=ax, fmt='.0f')
ax.set_title('Wine Production Heatmap — Top 10 Countries Over Time')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Stacked area chart — market share of top 5 over time
top5_data = df.head(5).set_index(country_col)[year_cols]
other_total = df.iloc[5:][year_cols].sum()
top5_data.loc['Others'] = other_total

fig, ax = plt.subplots(figsize=(14, 6))
top5_data.T.plot.area(ax=ax, alpha=0.7, 
                       color=['#722f37', '#c9b037', '#2ecc71', '#3498db', '#e74c3c', '#95a5a6'])
ax.set_title('Wine Production Market Share Over Time')
ax.set_ylabel('Production')
ax.legend(bbox_to_anchor=(1.05, 1))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Box plot of production distributions for top 10
top10_melted = df.head(10).melt(id_vars=country_col, value_vars=year_cols,
                                 var_name='Period', value_name='Production')

fig, ax = plt.subplots(figsize=(14, 6))
sns.boxplot(data=top10_melted, x=country_col, y='Production', ax=ax, palette='Set2')
ax.set_title('Production Variability — Top 10 Countries')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

## Key Findings

1. **Extreme concentration** — A handful of countries (Italy, France, Spain) dominate global wine production
2. **Traditional producers lead** — European Old World countries maintain their production dominance
3. **New World growth** — Countries like Australia, Chile, and New Zealand have grown their production
4. **Production variability** — Weather-dependent agriculture shows year-to-year fluctuations
5. **Market structure** — The wine production market is moderately concentrated (oligopoly)
6. **Rankings are stable** — Top producers tend to maintain their positions over time

## Limitations

- **Production ≠ quality** — Volume data says nothing about wine quality or value
- **No consumption data** — Can't compare production vs consumption patterns
- **Aggregate data** — Country-level totals hide regional variation (e.g., Bordeaux vs Languedoc)
- **No grape variety info** — Different wine types (red, white, rosé) aren't distinguished
- **Data gaps** — Some countries may have incomplete historical records

## Recommendations / Next Steps

1. **Merge with consumption data** — Identify net exporters vs importers
2. **Add economic data** — Correlate production with wine export revenue
3. **Climate analysis** — Overlay temperature/rainfall data to explain production fluctuations
4. **Quality metrics** — Combine with wine rating data (Wine Spectator, Vivino)
5. **Forecasting** — Use ARIMA or Prophet for production forecasting

### How to Extend This Analysis

- Analyze per-capita wine production by merging with population data
- Study the relationship between vineyard area and production output
- Build an interactive geographic visualization (choropleth map)

## Common Mistakes

1. **Equating production with quality/revenue** — A hectoliter of cheap table wine ≠ a hectoliter of Bordeaux
2. **Ignoring units** — Make sure you know whether data is in hectoliters, liters, or tonnes
3. **Not handling zeros** — Zero production for a period might mean missing data, not no production
4. **Not accounting for weather** — Year-to-year variation is natural in agriculture
5. **Assuming trends are permanent** — A few years of growth doesn't mean permanent market shift
6. **Ignoring the long tail** — Many small producers collectively contribute significantly

## Mini Challenge / Exercises

1. **Market concentration trend:** Calculate the CR5 (top 5 market share) for each period. Is the market becoming more or less concentrated?

2. **Growth champions:** Find countries that have more than doubled their production from the first to the last period.

3. **Correlation matrix:** Calculate pairwise correlations between the top 5 producers' production over time. Do they move together?

4. **Pareto analysis:** Calculate what percentage of countries produce 80% of the world's wine.

5. **Rank tracking:** Track how the top 5 rankings changed over the available periods.

In [ ]:
# Exercise space
# Exercise 4: Pareto analysis
# sorted_prod = df['Total'].sort_values(ascending=False)
# cum_pct = sorted_prod.cumsum() / sorted_prod.sum() * 100
# n_80 = (cum_pct <= 80).sum() + 1
# print(f"{n_80} out of {len(df)} countries produce 80% of wine ({n_80/len(df)*100:.1f}%)")

print("Uncomment the exercises above and try them!")

## Final Summary / Key Takeaways

| Aspect | Key Finding |
|--------|-------------|
| Dominance | Italy, France, Spain are the traditional leaders |
| Concentration | Top 5 countries produce majority of global wine |
| New World | Australia, Chile, South Africa growing their share |
| Variability | Production fluctuates due to weather/climate |
| Stability | Rankings remain relatively stable over time |
| Market Type | Oligopolistic structure |

Wine production is a fascinating lens through which to view global agriculture, trade, and cultural traditions. While the traditional European powerhouses maintain their dominance, New World producers have carved out significant and growing market shares, reshaping the global wine landscape.